This question involves the use of multiple linear regression on the `Auto` dataset.

In [ ]:
import statsmodels.api as sm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# import dataset
cwd = Path().resolve()
auto_csv = cwd.parents[1] / "data" / "Auto.csv"
Auto = pd.read_csv(auto_csv, na_values=['?']).dropna().set_index("name")
Auto

### (a) Scatterplot Matrix

In [ ]:
# scatterplot matrix
pd.plotting.scatter_matrix(Auto, figsize=(10,10));

### (b) Correlation Matrix

In [ ]:
Auto.corr()
# we see strong correlation between displacement-cylinder, horsepower-cylinder, horsepower-displacement, weight-displacement

### (c) Multiple Linear Regression with `sm.OLS()`
Notes
- Since `origin` is a categorical variable, I decided to encode it with one-hot encoding. `cylinders` could also be a categorical but I decided to keep it as is, since the number of cylinders still makes sense. Having an origin of '2' vs an origin of '3' on the other hand, doesn't make sense at all.

Questions
1. Is there a relationship between the predictors and response? Use `anova_lm()` to answer the question.
    - I tried using `anova_lm()` but since it requires another model result to compare to I put a basic linear regression with horsepower as a predictor. The multiple linear regression does seem to work well with a low F statistic probability. Hence there is a relationship between the predictors and response.
2. Which predictors appear to have a statistically significant relationship to the response?
    - Looking at t-statistic probabilities, for a 95% level of significance, all predictors except `acceleration` seem to have a relationship with the response.
3. What does the coefficient for the `year` variable suggest?
    - The `year` variable has a 0.5993 coefficient, which means that the `mpg` for cars generally increase by 0.5993 every year.

In [ ]:
X = Auto.drop(['mpg'], axis=1)
# one hot encode 'origin'
# for val in sorted(X['origin'].unique())[:-1]:
#     X[f'origin[{val}]'] = (X['origin'] == val).astype(int)
# X = X.drop(['origin'], axis=1)
X = pd.get_dummies(X, columns=['origin'], drop_first=True, dtype=int)

# fit OLS model
y = Auto['mpg']
model = sm.OLS(y, X)
result = model.fit()
result.summary()

In [ ]:
result2 = sm.OLS(y, Auto["horsepower"]).fit()
sm.stats.anova_lm(result2, result)

### (d) Diagnostic plots

Residual plots show a pattern of non-linearity, since the fitted values near the start and end tend to have a positive residual

Residual plots identify a few outliers, especially at the tail end of the fitted values.

Leverage plot identifies only 1 relatively high leverage point.

In [ ]:
# diagnostic plots
# - residual plots
# - leverage plots
fig, ax = plt.subplots(ncols=3, figsize=(12, 4))
# residual plot
ax[0].scatter(result.fittedvalues, result.resid)
ax[0].set_ylabel('residual')
ax[0].set_xlabel('fitted values')
ax[0].axhline(0, c='k', ls='--')
# leverage plot
ax[1].scatter(np.arange(X.shape[0]), result.get_influence().hat_matrix_diag)
ax[1].set_ylabel('leverage')
ax[1].set_xlabel('index')
# studentized t
# rse = np.sqrt(result.scale)
# leverage = result.get_influence().hat_matrix_diag
# studentized_resid = result.resid / (rse * np.sqrt(1 - leverage))
studentized_resid = result.get_influence().resid_studentized_internal
ax[2].scatter(result.fittedvalues, studentized_resid)
ax[2].set_ylabel('studentized residual')
ax[2].set_xlabel('fitted values')
ax[2].axhline(0, c='k', ls='--')
ax[2].axhline(3, c='r', ls='--')
ax[2].axhline(-3, c='r', ls='--')
plt.tight_layout()

### (e) Fit some models with interactions as described in the lab. Do any interactions appear significant?

Notes
- Fitted interactions in the presence of all other variables
- horsepower:weight seems to have the best effect, increasing R^2 relatively more than the others and producing a better looking residual plot, and thus seems to correct for non-linearity.
- I note that horsepower:weight seems to have a coefficient near to zero, but the 95% confidence interval does not contain zero, and the actual numerical values of this interaction term is very large, making the coefficient smaller. 

In [ ]:
# interactions
# cylinders	displacement	horsepower	weight	acceleration	
# we will try:
# - cylinders:displacement
# - cylinders:horsepower
# - horsepower:weight
# - displacement:weight
# - acceleration:weight
# - horsepower:acceleration

fig, axs = plt.subplots(figsize=(24, 16), ncols=3, nrows=2)

X_int = X.copy()
# X_int = X_int.drop(['year', 'origin_2', 'origin_3'], axis=1)
result_int_base = sm.OLS(y, X_int).fit()
for idx, int_term in enumerate([
    'cylinders:displacement',
    'cylinders:horsepower',
    'horsepower:weight',
    'displacement:weight',
    'acceleration:weight',
    'horsepower:acceleration',
]):
    col1, col2 = int_term.split(":")
    X_int_other = X_int.copy()
    X_int_other[int_term] = X_int_other[col1] * X_int_other[col2]
    result_with_int = sm.OLS(y, X_int_other).fit()
    anova_lm_result = sm.stats.anova_lm(result_int_base, result_with_int)
    print(f"int term: {int_term}")
    print(anova_lm_result)
    print(f"improvement in R^2: {result_int_base.rsquared} -> {result_with_int.rsquared}")

    # plot residual graph
    i = idx // 3
    j = idx % 3
    axs[i][j].scatter(result_with_int.fittedvalues, result_with_int.get_influence().resid_studentized_internal)
    axs[i][j].axhline(0, c='k', ls='--')
    axs[i][j].set_title(f"interaction: {int_term}")

plt.tight_layout()
# it seems that acceleration:weight and horsepower:acceleration help the model the most

In [ ]:
X_int_final = X.copy()
X_int_final["horsepower:weight"] = X_int_final["horsepower"] * X_int_final["weight"]
result_int_final = sm.OLS(y, X_int_final).fit()
result_int_final.summary()

### (f) Try a few different transformations of the variables

In presence of horsepower:weight intraction variable
- Investigated applying `log(X)`, `sqrt(X)`, and `pow(X, 2)` to `displacement`, `horsepower`, and `weight` variables
- No discernible improvement, in fact the model fared worse for the `horsepower` and `weight` variables.
- This is in the presence of the horsepower:weight interaction variable introduced earlier.

In absence of interaction variable
- Further investigation was done in the absence of the horsepower:weight interaction variable.
- Improvements in R^2 was seen in a few, but no improvement in the residual plots was observed.
- This shows that the transformations don't actually help.

Maybe a different transformation 1/X?
- I also wanted to know if doing `1/X` would help, but in fact it fared similarly to the other transformations and did not have much effect at all.

In [ ]:
# we will investigate applying `np.log`, `np.sqrt`, and `np.pow` to the displacement, horsepower, and weight variables

fig, axs = plt.subplots(ncols=4, nrows=3, figsize=(16,12))
for i, col in enumerate(['displacement', 'horsepower', 'weight']):
    for j in range(4):
        X_new = X_int_final.copy()
        if j == 0:
            X_new[col] = np.log(X_new[col])
        elif j == 1:
            X_new[col] = np.sqrt(X_new[col])
        elif j == 2:
            X_new[col] = np.pow(X_new[col], 2)
        else:
            X_new[col] = 1 / X_new[col]
    
        result_transform = sm.OLS(y, X_new).fit()
        anova_lm_result = sm.stats.anova_lm(result_int_final, result_transform)
        print(anova_lm_result)
        print(f"improvement in R^2: {result_int_final.rsquared} -> {result_transform.rsquared}")

        # plot residual graph
        axs[i][j].scatter(result_transform.fittedvalues, result_transform.get_influence().resid_studentized_internal)
        axs[i][j].axhline(0, c='k', ls='--')
        axs[i][j].set_title(f"col: {col}, transformation: {j}")


In [ ]:
# we will investigate applying `np.log`, `np.sqrt`, and `np.pow` to the displacement, horsepower, and weight variables
# in the ABSENCE of the interaction variable horsepower:weight

X_no_int = X_int_final.drop(['horsepower:weight'], axis=1)
result_no_int = sm.OLS(y, X_no_int).fit()


fig, axs = plt.subplots(ncols=4, nrows=3, figsize=(16,12))
for i, col in enumerate(['displacement', 'horsepower', 'weight']):
    for j in range(4):
        X_new = X_no_int.copy()
        if j == 0:
            X_new[col] = np.log(X_new[col])
        elif j == 1:
            X_new[col] = np.sqrt(X_new[col])
        elif j == 2:
            X_new[col] = np.pow(X_new[col], 2)
        else:
            X_new[col] = 1 / X_new[col]

    
        result_transform = sm.OLS(y, X_new).fit()
        anova_lm_result = sm.stats.anova_lm(result_no_int, result_transform)
        print(anova_lm_result)
        print(f"improvement in R^2: {result_no_int.rsquared} -> {result_transform.rsquared}")

        # plot residual graph
        axs[i][j].scatter(result_transform.fittedvalues, result_transform.get_influence().resid_studentized_internal)
        axs[i][j].axhline(0, c='k', ls='--')
        axs[i][j].set_title(f"col: {col}, transformation: {j}")